# InfoCuria event log analysis

This notebook prepares an event log for PM4Py, Disco and similar process-mining tools.
It reads the completed CSV produced by `01_dataset_downloader.py`, selects the useful event and case attributes,
and normalises activity labels. ECLI identifiers and document authors are kept in dedicated columns,
preserving document-type qualifiers such as `(Summary)` and `(OJ)`.
The exported `Country` column uses the single original or procedural language code resolved by the downloader.
All events of a `CaseID` share that value; genuinely multilingual cases use `MULTI`.
`Date_Lodging` preserves the official date of lodging of that proceeding in `YYYY-MM-DD` format,
repeated on every event, with empty values where the source does not provide a date.

PM4Py calculates the minimum and maximum event dates, the number of activity-sequence variants,
and the mean, median, maximum and minimum case durations in days. The notebook also calculates the mean
and median number of events per case and counts distinct cases
for the Court of Justice (`C`), General Court (`T`) and Civil Service Tribunal (`F`).
Each complete `CaseID` identifies a process-mining case, including procedural suffixes such as `T-201/04 R`.
Durations span the first and last recorded events of that identifier.
`MainCaseID`, when available, remains an attribute for linking related proceedings.

The notebook counts and excludes rows lacking a case identifier, activity or valid timestamp.
Source date anomalies are flagged. Only complete cases whose event timestamp years all fall
between `START_YEAR` and `END_YEAR`, inclusive, are retained for statistics and export.
Every event of a qualifying `CaseID` is preserved; cases extending outside the interval are excluded.
It creates `event_log` when required and exports the selected columns to `event_log/curia_log_en_START_YEAR-END_YEAR.csv`
using UTF-8 encoding and a semicolon delimiter. The same log is also saved in standard XES format
as `event_log/curia_log_en_START_YEAR-END_YEAR.xes` through PM4Py. Three category subsets are also exported
in both formats, with `_C`, `_T` and `_F` inserted before the final year suffix.
The year values in event log filenames come from the configuration constants.
PM4Py statistics for the complete log and each category are also saved in `event_log/_stats_START_YEAR-END_YEAR.csv`,
keyed by their CSV filenames, and category statistics are displayed during export.
A single `event_log/_activities_START_YEAR-END_YEAR.csv` also lists the distinct final activities
for the complete log and each category, with a total and alphabetically numbered list per CSV filename.

Run all cells with the project's Python 3.12 `.venv` kernel, from the directory containing this notebook.
The input path is set in the configuration cell below. Use `data/curia_scraping_en.csv`
for the completed collection, or select another collection or a sample.

In [1]:
# Import libraries used throughout the notebook.
import re
from pathlib import Path

import pandas as pd
import pm4py
from IPython.display import display
from pm4py.algo.querying.llm.abstractions.log_to_variants_descr import (
    compute_perf_aggregation,
)




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [2]:
# Inclusive timestamp-year limits: retain complete cases, preserving their histories.
START_YEAR = 2016
END_YEAR = 2026
if (type(START_YEAR) is not int or type(END_YEAR) is not int
        or not 1 <= START_YEAR <= END_YEAR <= 9999):
    raise ValueError("Years must be integers satisfying 1 <= START_YEAR <= END_YEAR <= 9999.")

# Read the completed CSV produced by the downloader.
PROJECT_DIR = Path.cwd().resolve()
INPUT_FILE = PROJECT_DIR / "data" / "curia_scraping_en.csv"
if not INPUT_FILE.exists():
    INPUT_FILE = PROJECT_DIR / "data" / "curia_scraping_en_partial.csv"
# Create the destination directory for the CSV and XES logs.
OUTPUT_DIR = PROJECT_DIR / "event_log"
OUTPUT_FILE = OUTPUT_DIR / f"curia_log_en_{START_YEAR}-{END_YEAR}.csv"
STATS_FILE = OUTPUT_DIR / f"_stats_{START_YEAR}-{END_YEAR}.csv"
ACTIVITIES_FILE = OUTPUT_DIR / f"_activities_{START_YEAR}-{END_YEAR}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / ".gitkeep").touch(exist_ok=True)

# Keep the event roles and attributes used for analysis and traceability.
EXPORT_COLUMNS = [
    "CaseID", "Activity", "Timestamp", "Case_Category", "MainCaseID", "Date_Lodging",
    "Activity_Party", "Country", "Area_of_Law",
    "ECLI", "Party_Names", "Date_Quality",
]


## Prepare the event log

Activities retain document-type qualifiers such as `(Summary)` and `(OJ)`.
ECLI identifiers and document authors are stored in `ECLI` and `Activity_Party`.
Rows missing a case identifier, activity or valid timestamp are counted and excluded.
Source date anomalies are flagged. A `CaseID` is retained only when all its valid event timestamps
have years between `START_YEAR` and `END_YEAR`, inclusive. All events of a qualifying case remain;
a case with any dated event outside the interval is excluded in full. Year filtering uses timestamps,
independently of the case number or legal closure year, and runs before statistics and exports.
Separate documents with the same case, activity and date remain separate events.

`Country` is read directly from the corrected downloader output. It contains one language code per
`CaseID`, repeated on every event: for example, `Avis 1/15` uses `FR` and `C-11/18 P` uses `EN`.
The downloader reads the original language for advisory cases and the procedural language for ordinary cases.
When the source remains genuinely multilingual, the value is `MULTI`.

The notebook checks that every case has exactly one valid, non-empty Country value before analysis or export.
An old CSV containing country names, multiple codes or inconsistent values must be repaired with
`01_dataset_downloader.py --enrich-countries CSV` or regenerated with the downloader.
`Country` is the only country or language attribute in the prepared log and all CSV/XES exports.

`Date_Lodging` comes directly from InfoCuria's `introductionDate` for each proceeding.
It identifies the official lodging date rather than the first document event, and remains an ISO date
(`YYYY-MM-DD`) in CSV and XES. All events of each internal `ProcedureID` share the same value. Different internal proceedings can
share one published `CaseID` while retaining different official lodging dates. Missing lodging dates
remain empty and do not exclude events. The notebook validates supplied dates and their consistency
within each non-empty `ProcedureID`; it never infers a lodging date from
the case number, main case or event timestamps. Year selection continues to use event timestamps.
Regenerate an older source CSV with the updated downloader to add this column from its cache.


In [3]:
input_columns = set(EXPORT_COLUMNS) | {"Area of Law", "ProcedureID"}
source = pd.read_csv(
    INPUT_FILE, sep=";", dtype="string", keep_default_na=False,
    usecols=lambda column: column in input_columns,
)
required_columns = {"CaseID", "Activity", "Timestamp", "Country", "Date_Lodging"}
missing_columns = required_columns.difference(source.columns)
if missing_columns:
    raise ValueError(
        f"Missing required event log columns: {sorted(missing_columns)}. "
        "Regenerate this CSV with the updated 01_dataset_downloader.py; cached case details are reused."
    )

if "Area_of_Law" not in source and "Area of Law" in source:
    source = source.rename(columns={"Area of Law": "Area_of_Law"})
# Country is already resolved by the downloader; never infer it from old aliases.
source["Country"] = source["Country"].str.strip().str.upper()
valid_countries = {
    "BG", "CS", "DA", "DE", "EL", "EN", "ES", "ET", "FI", "FR", "GA", "HR",
    "HU", "IT", "LT", "LV", "MT", "NL", "PL", "PT", "RO", "SK", "SL", "SV", "MULTI",
}
invalid_country = ~source["Country"].isin(valid_countries)
if invalid_country.any():
    examples = source.loc[invalid_country, ["CaseID", "Country"]].head(10).to_dict("records")
    raise ValueError(
        f"Invalid or missing Country values: {examples}. "
        "Repair this CSV with 01_dataset_downloader.py --enrich-countries CSV before exporting."
    )
country_counts = source.groupby(source["CaseID"].str.strip())["Country"].nunique()
conflicting_cases = country_counts[country_counts.ne(1)].index.tolist()
if conflicting_cases:
    raise ValueError(f"Each CaseID must have exactly one Country value: {conflicting_cases[:10]}")

for column in ("ECLI", "Activity_Party", "Date_Quality"):
    if column not in source:
        source[column] = ""
    source[column] = source[column].fillna("").str.strip()

source["CaseID"] = source["CaseID"].str.strip()
# Preserve the source lodging date as a case attribute, independently of event dates.
source["Date_Lodging"] = source["Date_Lodging"].fillna("").str.strip()
lodging_dates = pd.to_datetime(source["Date_Lodging"], format="%Y-%m-%d", errors="coerce")
invalid_lodging_dates = source["Date_Lodging"].ne("") & lodging_dates.isna()
if invalid_lodging_dates.any():
    examples = source.loc[invalid_lodging_dates, ["CaseID", "Date_Lodging"]].head(10).to_dict("records")
    raise ValueError(f"Invalid Date_Lodging values; expected YYYY-MM-DD or empty: {examples}")
source["Date_Lodging"] = lodging_dates.dt.strftime("%Y-%m-%d").fillna("").astype("string")
# Distinct internal proceedings can share one published CaseID and have different dates.
if "ProcedureID" in source:
    source["ProcedureID"] = source["ProcedureID"].fillna("").str.strip()
    lodging_counts = source.loc[source["ProcedureID"].ne("")].groupby("ProcedureID")["Date_Lodging"].nunique()
    conflicting_lodging_procedures = lodging_counts[lodging_counts.gt(1)].index.tolist()
    if conflicting_lodging_procedures:
        raise ValueError(
            f"Each ProcedureID must have one consistent Date_Lodging value: {conflicting_lodging_procedures[:10]}"
        )
source["Activity"] = source["Activity"].str.replace(r"\s+", " ", regex=True).str.strip()
ecli_pattern = re.compile(r"ECLI:[A-Z]{2}:[A-Z0-9_]+:\d{4}:[A-Z0-9._-]+", re.IGNORECASE)
extracted_ecli = source["Activity"].str.extract(f"({ecli_pattern.pattern})", flags=re.IGNORECASE, expand=False)
source["ECLI"] = source["ECLI"].mask(source["ECLI"].eq(""), extracted_ecli.fillna(""))
source["Activity"] = source["Activity"].str.replace(ecli_pattern, "", regex=True).str.strip()

# Recover observation authors from older CSV files.
author_labels = source["Activity"].str.extract(
    r"^(?P<activity>(?:Request for a preliminary ruling|Statement of case or written observations)"
    r"(?: - (?:Corrigendum|Addendum))?)\s*\((?P<author>[^()]+)\)\s*$"
)
source["Activity_Party"] = source["Activity_Party"].mask(
    source["Activity_Party"].eq(""), author_labels["author"].fillna("")
)
source["Activity"] = source["Activity"].where(author_labels["activity"].isna(), author_labels["activity"])
source["Activity"] = pd.Series(
    [activity.removesuffix(f" ({author})").strip()
     for activity, author in zip(source["Activity"], source["Activity_Party"])],
    index=source.index, dtype="string",
)
source["Activity"] = source["Activity"].str.replace(r"\s+", " ", regex=True).str.strip()

source["Timestamp"] = pd.to_datetime(source["Timestamp"], format="mixed", errors="coerce", utc=True)
future_dates = source["Timestamp"].gt(pd.Timestamp.now(tz="UTC"))
source.loc[future_dates & source["Date_Quality"].eq(""), "Date_Quality"] = "future_source_date"
# Use the procedure court for case numbers such as Avis 1/15.
case_category = source["CaseID"].str.extract(r"^([CTF])[-\s]", expand=False)
if "ProcedureID" in source:
    procedure_court = source["ProcedureID"].str.extract(r"^([CTF])/", expand=False)
    case_category = case_category.fillna(procedure_court)
source["Case_Category"] = case_category.fillna("Unknown")

missing_case = source["CaseID"].eq("")
missing_activity = source["Activity"].eq("")
invalid_timestamp = source["Timestamp"].isna()
valid_events = ~(missing_case | missing_activity | invalid_timestamp)

# Select whole cases by their first and last valid source timestamps.
# A dated event outside the interval excludes its case, even if its activity is missing.
case_timestamp_bounds = (
    source.loc[~missing_case & ~invalid_timestamp]
    .groupby("CaseID")["Timestamp"].agg(["min", "max"])
)
cases_in_year_range = (
    case_timestamp_bounds["min"].dt.year.ge(START_YEAR)
    & case_timestamp_bounds["max"].dt.year.le(END_YEAR)
)
selected_case_ids = case_timestamp_bounds.index[cases_in_year_range]
case_in_year_range = source["CaseID"].isin(selected_case_ids)
events_excluded_by_year_range = valid_events & ~case_in_year_range
valid_events = valid_events & case_in_year_range
source["_source_order"] = range(len(source))
selected_columns = [column for column in EXPORT_COLUMNS if column in source]
event_log = (
    source.loc[valid_events]
    .sort_values(["CaseID", "Timestamp", "_source_order"])
    .loc[:, selected_columns]
    .reset_index(drop=True)
)
if event_log.empty:
    raise ValueError(
        f"No events remain after validation and complete-case filtering for {START_YEAR}–{END_YEAR}."
    )

quality_summary = pd.Series({
    "Input events": len(source),
    "Rows with missing case identifiers": int(missing_case.sum()),
    "Rows with missing activities": int(missing_activity.sum()),
    "Rows with invalid timestamps": int(invalid_timestamp.sum()),
    "Cases before year filtering": len(case_timestamp_bounds),
    "Cases excluded by year range": int((~cases_in_year_range).sum()),
    "Events excluded by year range": int(events_excluded_by_year_range.sum()),
    "Excluded events": int((~valid_events).sum()),
    "Retained events": len(event_log),
    "Events with flagged source dates": int(event_log["Date_Quality"].ne("").sum()),
    "Events with lodging dates": int(event_log["Date_Lodging"].ne("").sum()),
    "Cases without lodging dates": int(event_log.loc[event_log["Date_Lodging"].eq(""), "CaseID"].nunique()),
    "Events with Country values": int(event_log["Country"].ne("").sum()),
    "Cases with a single Country value": int(event_log["CaseID"].nunique()),
    "Cases marked MULTI": int(event_log.loc[event_log["Country"].eq("MULTI"), "CaseID"].nunique()),
}, name="Count").to_frame()
print(f"Complete cases with all event years in {START_YEAR}–{END_YEAR} (inclusive)")
display(quality_summary)
display(event_log.head(10))

Complete cases with all event years in 2016–2026 (inclusive)


,Count
Input events,72361
Rows with missing case identifiers,0
Rows with missing activities,0
Rows with invalid timestamps,0
Cases before year filtering,18841
Cases excluded by year range,1989
Events excluded by year range,7776
Excluded events,7776
Retained events,64585
Events with flagged source dates,0


,CaseID,Activity,Timestamp,Case_Category,MainCaseID,Date_Lodging,Activity_Party,Country,Area_of_Law,ECLI,Party_Names,Date_Quality
0,Avis 1/17,Opinion (OJ),2017-10-13 00:00:00+00:00,C,Avis 1/17,2017-09-07,,FR,Law governing the institutions | External acti...,,Accord ECG UE-Canada,
1,Avis 1/17,Opinion,2019-01-29 00:00:00+00:00,C,Avis 1/17,2017-09-07,,FR,Law governing the institutions | External acti...,ECLI:EU:C:2019:72,Accord ECG UE-Canada,
2,Avis 1/17,Opinions of the Court,2019-04-30 00:00:00+00:00,C,Avis 1/17,2017-09-07,,FR,Law governing the institutions | External acti...,ECLI:EU:C:2019:341,Accord ECG UE-Canada,
3,Avis 1/17,Abstract,2019-04-30 00:00:00+00:00,C,Avis 1/17,2017-09-07,,FR,Law governing the institutions | External acti...,ECLI:EU:C:2019:341,Accord ECG UE-Canada,
4,Avis 1/17,Opinion (OJ),2019-06-14 00:00:00+00:00,C,Avis 1/17,2017-09-07,,FR,Law governing the institutions | External acti...,,Accord ECG UE-Canada,
5,Avis 1/19,Opinion (OJ),2019-11-22 00:00:00+00:00,C,Avis 1/19,2019-07-09,,EN,External action,,Convention d’Istanbul,
6,Avis 1/19,Opinion,2021-03-11 00:00:00+00:00,C,Avis 1/19,2019-07-09,,EN,External action,ECLI:EU:C:2021:198,Convention d’Istanbul,
7,Avis 1/19,Opinions of the Court,2021-10-06 00:00:00+00:00,C,Avis 1/19,2019-07-09,,EN,External action,,Convention d’Istanbul,
8,Avis 1/20,Opinion (OJ),2021-01-29 00:00:00+00:00,C,Avis 1/20,2020-12-02,,MULTI,Law governing the institutions,,"Avis rendu en vertu de l’article 218, paragrap...",
9,Avis 1/20,Opinions of the Court,2022-06-16 00:00:00+00:00,C,Avis 1/20,2020-12-02,,MULTI,Law governing the institutions,ECLI:EU:C:2022:485,"Avis rendu en vertu de l’article 218, paragrap...",


In [4]:
# Show every distinct activity before renaming.
print("Distinct activities before renaming:")
print("Total:", event_log["Activity"].dropna().nunique())
print("\n".join(sorted(event_log["Activity"].dropna().unique())))


Distinct activities before renaming:
Total: 27
Abstract
Application (OJ)
Decision to review
Decision to review (OJ)
Judgment
Judgment (Information)
Judgment (OJ)
Judgment (Summary)
Judgment (extracts)
Opinion
Opinion (OJ)
Opinions of the Court
Order
Order (Information)
Order (OJ)
Order (Summary)
Order (extracts)
Press release
Removal (OJ)
Request for a preliminary ruling
Request for a preliminary ruling - Addendum
Request for a preliminary ruling - Corrigendum
Ruling (Summary)
Statement of case or written observations
Statement of case or written observations - Addendum
Statement of case or written observations - Corrigendum
View


In [5]:
# Rename only the exact activity label Opinion.
event_log.loc[event_log["Activity"].eq("Opinion"), "Activity"] = "Opinion (AG)"
print("Distinct activities after renaming:")
print("Total:", event_log["Activity"].dropna().nunique())
print("\n".join(sorted(event_log["Activity"].dropna().unique())))


Distinct activities after renaming:
Total: 27
Abstract
Application (OJ)
Decision to review
Decision to review (OJ)
Judgment
Judgment (Information)
Judgment (OJ)
Judgment (Summary)
Judgment (extracts)
Opinion (AG)
Opinion (OJ)
Opinions of the Court
Order
Order (Information)
Order (OJ)
Order (Summary)
Order (extracts)
Press release
Removal (OJ)
Request for a preliminary ruling
Request for a preliminary ruling - Addendum
Request for a preliminary ruling - Corrigendum
Ruling (Summary)
Statement of case or written observations
Statement of case or written observations - Addendum
Statement of case or written observations - Corrigendum
View


## PM4Py statistics

A variant is an ordered sequence of activity labels.
Events sharing a date keep their input order, because the source provides day-level precision.
Durations use elapsed calendar time between the first and last recorded events of each `CaseID`.
This span includes subsequent publications and retained source date anomalies.

Event counts are calculated per complete `CaseID`; repeated activities count as separate events.
The mean and median summarise these per-case event counts.

Timestamp values and per-case event counts come from `pm4py.get_event_attribute_values`, case durations from
`pm4py.get_all_case_durations`, and variants from `pm4py.get_variants_as_tuples`.
PM4Py's `compute_perf_aggregation` calculates timestamp bounds, event-count and duration aggregates.
For an even number of cases, the median averages the two central values.

In [6]:
def calculate_log_statistics(log_frame, file_name):
    """Calculate PM4Py case, event-count, variant, timestamp and duration statistics."""
    statistics = {
        "File": file_name,
        "Cases": 0,
        "Mean_Events_Per_Case": None,
        "Median_Events_Per_Case": None,
        "Variants": 0,
        "Minimum_Timestamp": None,
        "Maximum_Timestamp": None,
        "Mean_Duration_Days": None,
        "Median_Duration_Days": None,
        "Minimum_Duration_Days": None,
        "Maximum_Duration_Days": None,
    }
    if log_frame.empty:
        return statistics

    formatted_log = pm4py.format_dataframe(
        log_frame.copy(), case_id="CaseID", activity_key="Activity", timestamp_key="Timestamp"
    )
    case_event_counts = pm4py.get_event_attribute_values(formatted_log, "case:concept:name")
    statistics["Cases"] = len(case_event_counts)
    event_counts = list(case_event_counts.values())
    for column, aggregation in (("Mean_Events_Per_Case", "mean"), ("Median_Events_Per_Case", "median")):
        statistics[column] = round(float(compute_perf_aggregation(event_counts, aggregation)), 2)
    statistics["Variants"] = len(pm4py.get_variants_as_tuples(formatted_log))
    timestamp_values = pm4py.get_event_attribute_values(formatted_log, "time:timestamp")
    timestamp_seconds = [pd.Timestamp(timestamp).timestamp() for timestamp in timestamp_values]
    for column, aggregation in (("Minimum_Timestamp", "min"), ("Maximum_Timestamp", "max")):
        statistics[column] = pd.to_datetime(
            compute_perf_aggregation(timestamp_seconds, aggregation), unit="s", utc=True
        ).isoformat()
    durations = pm4py.get_all_case_durations(formatted_log, business_hours=False)
    for column, aggregation in (
        ("Mean_Duration_Days", "mean"),
        ("Median_Duration_Days", "median"),
        ("Minimum_Duration_Days", "min"),
        ("Maximum_Duration_Days", "max"),
    ):
        statistics[column] = round(float(compute_perf_aggregation(durations, aggregation)) / 86400, 2)
    return statistics


complete_statistics = calculate_log_statistics(event_log, OUTPUT_FILE.name)
log_summary = pd.Series({
    "Minimum date": complete_statistics["Minimum_Timestamp"],
    "Maximum date": complete_statistics["Maximum_Timestamp"],
    "Events": len(event_log),
    "Distinct cases": complete_statistics["Cases"],
    "Mean events per case": f"{complete_statistics['Mean_Events_Per_Case']:.2f}",
    "Median events per case": f"{complete_statistics['Median_Events_Per_Case']:.2f}",
    "Distinct activities": event_log["Activity"].nunique(),
    "Variants": complete_statistics["Variants"],
}, name="Value").to_frame()
display(log_summary)


,Value
Minimum date,2016-01-08T00:00:00+00:00
Maximum date,2026-11-19T00:00:00+00:00
Events,64585
Distinct cases,16852
Mean events per case,3.83
Median events per case,4.00
Distinct activities,27
Variants,665


In [7]:
# The shared helper calculates these aggregates through PM4Py, in calendar days.
duration_summary = pd.DataFrame({
    "Duration (days)": {
        "Mean": complete_statistics["Mean_Duration_Days"],
        "Median": complete_statistics["Median_Duration_Days"],
        "Maximum": complete_statistics["Maximum_Duration_Days"],
        "Minimum": complete_statistics["Minimum_Duration_Days"],
    }
})
duration_summary.index.name = "Case duration statistic"
display(duration_summary.map(lambda value: f"{value:.2f}"))


,Duration (days)
Case duration statistic,
Mean,425.18
Median,402.50
Maximum,2950.00
Minimum,0.00


In [8]:
case_types = ["C", "T", "F"]
if event_log["Case_Category"].eq("Unknown").any():
    case_types.append("Unknown")
case_counts = (
    event_log[["CaseID", "Case_Category"]]
    .drop_duplicates()
    .groupby("Case_Category").size()
    .reindex(case_types, fill_value=0)
    .rename("Distinct cases").to_frame()
)
display(case_counts)

,Distinct cases
Case_Category,
C,7857
T,8916
F,79


## Export CSV and XES

The export writes the prepared log after complete-case year filtering and three subsets selected by `Case_Category`.
All eight files use the same inclusive `START_YEAR`–`END_YEAR` selection.
Each log is saved in CSV and XES with the same selected attributes.
The sole country or language attribute is `Country`, read from the corrected downloader output.
All events of each `CaseID` retain the same validated value in both formats.
`Date_Lodging` is also preserved on every event as the official proceeding lodging date
in `YYYY-MM-DD` format; empty source dates remain empty in both formats.
Every event log filename ends with `_{START_YEAR}-{END_YEAR}` before its extension.
Category filenames insert `_C`, `_T` or `_F` immediately before that year suffix.
With the configured output name, the files are

```text
curia_log_en_2016-2026.csv     curia_log_en_2016-2026.xes
curia_log_en_C_2016-2026.csv   curia_log_en_C_2016-2026.xes
curia_log_en_T_2016-2026.csv   curia_log_en_T_2016-2026.xes
curia_log_en_F_2016-2026.csv   curia_log_en_F_2016-2026.xes
```

A category with no events produces a CSV containing the column headers and an empty XES log.

Each category's PM4Py statistics are displayed beside its exports: case and variant counts,
mean and median numbers of events per case (`Mean_Events_Per_Case` and `Median_Events_Per_Case`),
minimum and maximum UTC timestamps, and mean, median, minimum and maximum case durations in days.
The complete log and all three categories also have one row each in `event_log/_stats_START_YEAR-END_YEAR.csv`,
using the CSV filename as the unique `File` key. CSV and XES versions share the same statistics.
The statistics filename includes the configured year interval, for example `_stats_2016-2026.csv`.
It uses UTF-8 and a semicolon delimiter. Re-running updates the matching filenames within that interval.
Duration values and mean/median event counts are rounded, displayed and saved to two decimal places;
case and variant counts remain integers. Empty categories have zero cases and variants,
with blank timestamp, duration and mean/median event-count values.

The single `event_log/_activities_START_YEAR-END_YEAR.csv` lists distinct activities for the complete log
and all three categories using their CSV filenames. Its columns are `File`, `Total_Activities`,
`Activity_Number` and `Activity`. Final activity labels (including `Opinion (AG)`) are sorted alphabetically,
with numbering starting at 1 for each log and its distinct-activity total repeated on every row.
An empty category has one row with total 0 and blank number and activity.
This UTF-8, semicolon-delimited report is replaced on each run for the selected year interval.

For CSV imports, map `CaseID` to the case identifier, `Activity` to the activity and `Timestamp` to the timestamp.
Use timestamp format `yyyy-MM-dd HH:mm:ss` and delimiter `;`.

PM4Py writes the XES case identifier as the trace's `concept:name`, activities as event `concept:name`
and UTC timestamps as `time:timestamp`. Other selected attributes remain on their events.
Missing optional source columns are omitted.

In [9]:
def save_activity_summary(logs, csv_file):
    """List each log's distinct final activities with totals and numbering from 1."""
    records = []
    for file_name, log_frame in logs.items():
        activities = sorted(log_frame["Activity"].dropna().unique())
        for number, activity in enumerate(activities or [""], start=1):
            records.append({
                "File": file_name,
                "Total_Activities": len(activities),
                "Activity_Number": number if activities else "",
                "Activity": activity,
            })
    activity_table = pd.DataFrame(
        records, columns=["File", "Total_Activities", "Activity_Number", "Activity"]
    )
    temporary_file = csv_file.with_suffix(".csv.tmp")
    activity_table.to_csv(temporary_file, sep=";", index=False, encoding="utf-8")
    temporary_file.replace(csv_file)
    print(f"Saved distinct activities for {len(logs)} logs to {csv_file}")


def save_event_log(log_frame, csv_file):
    """Write the prepared events to CSV with a semicolon delimiter and to XES."""
    export_log = log_frame.copy()
    export_log["Timestamp"] = export_log["Timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    temporary_file = csv_file.with_suffix(".csv.tmp")
    export_log.to_csv(temporary_file, sep=";", index=False, encoding="utf-8")

    # Use the XES standard roles while preserving the selected event attributes.
    xes_dataframe = log_frame.rename(columns={
        "CaseID": "case:concept:name",
        "Activity": "concept:name",
        "Timestamp": "time:timestamp",
    })
    xes_log = pm4py.convert_to_event_log(xes_dataframe, case_id_key="case:concept:name")
    xes_log.classifiers["Activity"] = ["concept:name"]
    xes_file = csv_file.with_suffix(".xes")
    temporary_xes_file = xes_file.with_name(xes_file.stem + ".tmp.xes")
    pm4py.write_xes(
        xes_log, str(temporary_xes_file), encoding="utf-8",
        variant_str="line_by_line", show_progress_bar=False,
    )

    # Publish each completed pair after both exports have been written.
    temporary_file.replace(csv_file)
    temporary_xes_file.replace(xes_file)
    print(f"Saved {len(export_log):,} events from {len(xes_log):,} cases to {csv_file}")
    print(f"Saved the XES event log to {xes_file}")
    print()

year_suffix = f"_{START_YEAR}-{END_YEAR}"
save_event_log(event_log, OUTPUT_FILE)
exported_logs = {OUTPUT_FILE.name: event_log}
statistics_records = [calculate_log_statistics(event_log, OUTPUT_FILE.name)]
for category in ("C", "T", "F"):
    category_log = event_log.loc[event_log["Case_Category"].eq(category)]
    category_file = OUTPUT_FILE.with_name(
        f"{OUTPUT_FILE.stem.removesuffix(year_suffix)}_{category}{year_suffix}{OUTPUT_FILE.suffix}"
    )
    save_event_log(category_log, category_file)
    exported_logs[category_file.name] = category_log
    category_statistics = calculate_log_statistics(category_log, category_file.name)
    statistics_records.append(category_statistics)
    print(f"PM4Py statistics: {category_file.name}")
    category_display = {
        field: ("" if value is None else f"{value:.2f}"
                if field.endswith(("_Days", "_Events_Per_Case")) else str(value))
        for field, value in category_statistics.items() if field != "File"
    }
    display(pd.Series(category_display, name="Value").to_frame())
    print()

# File is the unique key for each CSV/XES pair in this year interval.
current_statistics = pd.DataFrame(statistics_records).set_index("File")
if not current_statistics.index.is_unique:
    raise ValueError("Statistics must contain a unique File value for each log.")
if STATS_FILE.exists():
    previous_statistics = pd.read_csv(STATS_FILE, sep=";", index_col="File")
    previous_statistics = previous_statistics.loc[~previous_statistics.index.isin(current_statistics.index)]
    statistics_table = pd.concat([previous_statistics, current_statistics])
else:
    statistics_table = current_statistics
statistics_table = statistics_table.loc[~statistics_table.index.duplicated(keep="last")].sort_index()
temporary_stats_file = STATS_FILE.with_suffix(".csv.tmp")
statistics_table.to_csv(
    temporary_stats_file, sep=";", encoding="utf-8", index_label="File", float_format="%.2f"
)
temporary_stats_file.replace(STATS_FILE)
print(f"Saved statistics for {len(current_statistics)} logs to {STATS_FILE}")

save_activity_summary(exported_logs, ACTIVITIES_FILE)

/Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/.venv/lib/python3.12/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


Saved 64,585 events from 16,852 cases to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_2016-2026.csv
Saved the XES event log to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_2016-2026.xes

Saved 35,827 events from 7,857 cases to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_C_2016-2026.csv
Saved the XES event log to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_C_2016-2026.xes

PM4Py statistics: curia_log_en_C_2016-2026.csv


,Value
Cases,7857
Mean_Events_Per_Case,4.56
Median_Events_Per_Case,4.00
Variants,551
Minimum_Timestamp,2016-01-08T00:00:00+00:00
Maximum_Timestamp,2026-11-19T00:00:00+00:00
Mean_Duration_Days,423.84
Median_Duration_Days,462.00
Minimum_Duration_Days,0.00
Maximum_Duration_Days,2050.00



Saved 28,626 events from 8,916 cases to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_T_2016-2026.csv
Saved the XES event log to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_T_2016-2026.xes

PM4Py statistics: curia_log_en_T_2016-2026.csv


,Value
Cases,8916
Mean_Events_Per_Case,3.21
Median_Events_Per_Case,4.00
Variants,183
Minimum_Timestamp,2016-01-08T00:00:00+00:00
Maximum_Timestamp,2026-11-18T00:00:00+00:00
Mean_Duration_Days,429.91
Median_Duration_Days,364.00
Minimum_Duration_Days,0.00
Maximum_Duration_Days,2950.00



Saved 132 events from 79 cases to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_F_2016-2026.csv
Saved the XES event log to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/curia_log_en_F_2016-2026.xes

PM4Py statistics: curia_log_en_F_2016-2026.csv


,Value
Cases,79
Mean_Events_Per_Case,1.67
Median_Events_Per_Case,1.00
Variants,10
Minimum_Timestamp,2016-01-08T00:00:00+00:00
Maximum_Timestamp,2016-09-30T00:00:00+00:00
Mean_Duration_Days,24.97
Median_Duration_Days,0.00
Minimum_Duration_Days,0.00
Maximum_Duration_Days,224.00



Saved statistics for 4 logs to /Volumes/SAMSUNG-PHD/PhD/Articoli MIEI/CLSR Special Issue 2026 - CURIA e DECLARE/curia_gpai/00 - event_log/event_log/_stats_2016-2026.csv


PM4Py supplies timestamp values, per-case event counts, case durations and activity-sequence variants from a formatted copy of the event log.
Its aggregation function calculates timestamp bounds, mean and median event counts, and mean, median, maximum and minimum case durations.
The CSV and XES exports preserve the analysis attributes defined above.
[PM4Py statistics API](https://processintelligence.solutions/app/pm4py/api/api/pm4py.stats.html),
[PM4Py aggregation API](https://processintelligence.solutions/app/pm4py/api/api/pm4py.algo.querying.llm.abstractions.html#pm4py.algo.querying.llm.abstractions.log_to_variants_descr.compute_perf_aggregation),
[PM4Py formatting API](https://processintelligence.solutions/app/pm4py/api/api/pm4py.utils.html).
[PM4Py XES export API](https://processintelligence.solutions/app/pm4py/api/api/pm4py.write.html#pm4py.write.write_xes).